# **Data Collection**

## Objectives

* Fetch data from Kaggle and save as raw data
* Inspect the data and save to an outputs directory

## Inputs

* Kaggle JSON file - authentication token required

## Outputs

* Generate dataset, saved to outputs/datasets/collection/HotelBookings.csv

## Additional Comments

* Notebook developed in local environment, changes may be required for cloud based IDEs


---

# Change working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [ ]:
import os
current_dir = os.getcwd()
current_dir

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [ ]:
os.chdir(os.path.dirname(current_dir))

current_dir = os.getcwd()
current_dir

## Setup

* Install Kaggle (if not already installed)

In [ ]:
!uv pip install kaggle

* Configure Kaggle
    * Change the Kaggle directory to the current working directory
    * Set permissions for the kaggle authentication token

In [ ]:
os.environ['KAGGLE_CONFIG_DIR'] = os.getcwd()
! chmod 600 kaggle.json

## Retrieve Dataset

* Download the dataset .zip file

In [ ]:
! kaggle datasets download -d "jessemostipak/hotel-booking-demand" -p "inputs/datasets"

* Unzip the downloaded file, remove the .zip file and credentials file

In [ ]:
! unzip "inputs/datasets"/*.zip -d "inputs/datasets" \
    && rm "inputs/datasets"/*.zip \
    && rm kaggle.json

---

## Load and Inspect the Dataset

* Load the dataframe

In [ ]:
import pandas as pd
df = pd.read_csv(f"inputs/datasets/hotel_bookings.csv")
df.head()

---

* Dataframe summary

In [ ]:
print(df.shape)
print()
df.info()

* Check for empty values

In [ ]:
df.isnull().sum()

* Check for duplicate rows

In [ ]:
df[df.duplicated(keep=False)]

* Check for target variable imbalance

In [ ]:
target = df["is_canceled"]
target.value_counts(normalize=True)

## Time-shift the data to a more recent timeframe

* Examine cancellation distribution for each year of data

In [ ]:
import seaborn as sns
sns.countplot(data=df,
              x="arrival_date_year",
              hue="is_canceled")

* Although the cancellation rate seems balanced across the 3 years, the number of bookings is not. To preserve the integrity of the data, a simple renaming of the years is the most appropriate date shift.
* Dates to be replaced with 2023, 2024 and 2025

In [ ]:
df_updated_dates = df.replace(to_replace={
    2015: 2023,
    2016: 2024,
    2017: 2025
})

df_updated_dates["reservation_status_date"] = (
    df_updated_dates["reservation_status_date"]
    .str.replace('2015', '2023')
    .str.replace('2016', '2024')
    .str.replace('2017', '2025')
)

df_updated_dates.head()

## Conclusions
* The dataset details 119,390 bookings across 32 features providing a substantial sample for modelling
* There are null values in the following columns: (agent: 16340), (company, 112593), (country, 488), (children, 4)
* Duplicate row analysis showed 40165 duplicate rows. In the absence of booking identifier information, it cannot be ascertained that these are genuine duplicates. It is highly plausible that these represent group bookings or even coincidental feature sharing across the various hotels.
* 37% of bookings are cancelled, indicating a moderate class imbalance

---

# Push files to Repo

* Create folder to store the dataset

In [ ]:
import os
try:
  os.makedirs(name="outputs/datasets/collection")
except Exception as e:
  print(e)

df_updated_dates.to_csv(f"outputs/datasets/collection/HotelBookings.csv", index=False)